# Building Stock and Consumption Description

**Purpose:** Descriptive statistics of the SDES-2018 building stock — composition,
energy performance distribution, thermal transmittance, and consumption analysis.

**Project imports:** `project.input.resources`, `project.model`, `project.utils`.

**Inputs:** Building stock (via `get_inputs`).

**Outputs:** Stock distribution figures, cross-tabs, consumption statistics.

**Consolidates:** Previously split across `description_stock.ipynb` and `description_consumption.ipynb`.

In [ ]:
import os
import pandas as pd
"""import sys
project_path = os.path.join(*([os.sep]+os.path.normpath(os.getcwd()).split(os.sep)[:5]))
if project_path not in sys.path: sys.path.append(project_path)"""

from project.input.resources import resources_data
from project.model import get_inputs
from project.utils import subplots_attributes, subplots_pie, plot_attribute, plot_attribute2attribute
from project.utils import cumulated_plot,cumulated_plots


In [ ]:
output = 'output/description_stock'
if not os.path.isdir(output):
    os.mkdir(output)

## Loading inputs
Inputs come from the Res-IRF reference scenario.

In [ ]:
inputs = get_inputs(variables=['buildings'])
buildings = inputs['buildings']

## General description

In [ ]:
stock = buildings.simplified_stock(energy_level=True)
stock = stock.groupby(['Occupancy status', 'Income owner', 'Income tenant', 'Housing type', 'Energy', 'Performance']).sum()

In [ ]:
subplots_attributes(stock, dict_order=resources_data['index'], dict_color=resources_data['colors'], percent=False, sharey=True)
subplots_attributes(stock, dict_order=resources_data['index'], dict_color=resources_data['colors'], percent=False, sharey=True, save=os.path.join(output, 'stock_sdes2018_all.png'))

In [ ]:
subplots_pie(stock, dict_order=resources_data['index'], pie=['Housing type', 'Energy', 'Occupancy status'], dict_color=resources_data['colors'], percent=False)


In [ ]:
plot_attribute(stock, attribute='Performance', dict_order=resources_data['index'], percent=False, dict_color=resources_data['colors'], width=0.4,
               save='output/stock_sdes2018_energy_performance.png', figsize=(8.0, 6.0))

In [ ]:
plot_attribute2attribute(stock, 'Performance', 'Energy', dict_order=resources_data['index'], dict_color=resources_data['colors'])
plot_attribute2attribute(stock, 'Performance', 'Energy', dict_order=resources_data['index'], dict_color=resources_data['colors'], percent=False)
plot_attribute2attribute(stock, 'Performance', 'Energy', dict_order=resources_data['index'], dict_color=resources_data['colors'], percent=False, save='output/stock_sdes2018_dpe_energy.png')

In [ ]:
plot_attribute2attribute(stock, 'Performance', 'Energy', dict_order=resources_data['index'], dict_color=resources_data['colors'], percent=True)

### Table

Energy by energy performance certificate

In [ ]:
stock.groupby(['Performance', 'Energy']).sum().unstack('Energy')

Energy by energy performance certificate (%)

In [ ]:
(stock.groupby(['Performance', 'Energy']).sum().unstack('Energy').T / stock.groupby(['Performance', 'Energy']).sum().unstack('Energy').sum(axis=1)).T

## Description of the thermal performance

In [ ]:
temp = dict()
for i in ['Wall', 'Floor', 'Roof', 'Windows']:
    y = pd.Series(buildings.stock.index.get_level_values(i), index=buildings.stock.index, name='{} insulation (W/m2.K)'.format(i)).astype('float')
    x = buildings.stock / 10**6
    temp.update({i: cumulated_plot(x, y, plot=False)})
cumulated_plots(temp, 'Thermal transmittance U (W/m2.K)', ylim=3)

In [ ]:
cumulated_plots(temp, 'Thermal transmittance U (W/m2.K)', ylim=3)

In [ ]:
stock.groupby(['Housing type', 'Occupancy status', 'Performance']).sum().to_csv(os.path.join(output, 'stock_grouped.csv'))

## Consumption analysis

Consumption statistics derived from the building stock. Requires `output` from a
`mitigation_potential()` call or equivalent.

In [ ]:
# Load mitigation output for consumption data
from project.utils import cumulated_plot, cumulated_plots

full_inputs = get_inputs(path='output/static')
full_buildings = full_inputs['buildings']
output_mitigation = full_buildings.mitigation_potential(
    full_inputs['energy_prices'], full_inputs['cost_insulation'],
    full_inputs['carbon_emission'], full_inputs['carbon_value_kwh']
)

In [ ]:
total_surface = (buildings.surface * buildings.stock).sum()
print('Surface of the building stock: {:,.0f} Million m2'.format(total_surface / 10**6))

consumption_total_before = output_mitigation['Consumption actual before (kWh/segment)'].sum()
print('Actual space heating: {:,.0f} TWh'.format(consumption_total_before / 10**9))

consumption_standard_before = output_mitigation['Consumption before (kWh/segment)'].sum()
print('Conventional space heating: {:,.0f} TWh'.format(consumption_standard_before / 10**9))

### kWh/m2 distribution

In [ ]:
print('Average actual: {:.0f} kWh/m2'.format(
    output_mitigation['Consumption actual before (kWh/segment)'].sum() / total_surface))
print('Average conventional: {:.0f} kWh/m2'.format(
    output_mitigation['Consumption before (kWh/segment)'].sum() / total_surface))

(output_mitigation['Consumption before (kWh/dwelling)'] / buildings.surface).describe()

### kWh/dwelling distribution

In [ ]:
print(output_mitigation['Consumption actual before (kWh/dwelling)'].describe())